# 第一阶段 步骤07：反向传播的自动化

> 来源：《深度学习入门2：自制框架》（斋藤康毅 著，郑明智 译，人民邮电出版社 2023）
> 目标：从零构建深度学习框架 **DeZero** 的第七步。

---

## 核心目标

步骤6 的反向传播还得**手动写调用顺序**，函数一多就易错。本步让反向传播**自动化**：给变量和函数之间建立"连接"，程序就能自动沿计算图回溯。

这套"边计算边建图"的机制，就是 **Define-by-Run（动态计算图）** 的核心。

## 7.1 为反向传播的自动化创造条件

核心思想：**变量是由函数"创造"的**——函数是变量的"父母"（creator）。

- 给 `Variable` 加 `creator` 属性 + `set_creator` 方法；
- `Function` 在创建输出时，让输出记住"创造我的是谁"（`output.set_creator(self)`）。

建立连接后，正、反向就能互通：

| 通过 | 找到 |
| --- | --- |
| `Variable.creator` | 创造它的函数（往前一步） |
| `Function.input` | 函数的输入变量（再往前一步） |

两步交替，就能沿计算图反向遍历。

In [ ]:
import numpy as np

# Variable：新增 creator（记录"谁创造了我"）
class Variable:
    def __init__(self, data):
        self.data = data
        self.grad = None
        self.creator = None       # 创造我的函数

    def set_creator(self, func):
        self.creator = func

# Function：正向传播时让输出记录创造者，并保存输入/输出
class Function:
    def __call__(self, input):
        x = input.data
        y = self.forward(x)
        output = Variable(y)
        output.set_creator(self)  # 输出记住：创造我的是这个函数
        self.input = input        # 保存输入
        self.output = output      # 保存输出
        return output

    def forward(self, x):
        raise NotImplementedError()

    def backward(self, gy):
        raise NotImplementedError()

class Square(Function):
    def forward(self, x):
        return x ** 2
    def backward(self, gy):
        x = self.input.data
        return 2 * x * gy

class Exp(Function):
    def forward(self, x):
        return np.exp(x)
    def backward(self, gy):
        x = self.input.data
        return np.exp(x) * gy

## 7.2 尝试反向传播

有了连接，就能从输出出发，反复执行**三步**反向回溯：

1. 获取函数（`y.creator`）；
2. 获取函数的输入（`f.input`）；
3. 调用函数的 `backward`（`f.backward(gy)`）。

三步循环往复，直到碰到 `creator` 为 `None` 的变量（用户直接提供的输入）。

In [ ]:
A = Square()
B = Exp()
C = Square()

x = Variable(np.array(0.5))
a = A(x)
b = B(a)
y = C(b)

# 验证"连接"是否正确建立
assert y.creator == C
assert y.creator.input == b
assert y.creator.input.creator == B

# 沿连接手动反向回溯（三步循环）
y.grad = np.array(1.0)
C = y.creator                    # 1. 获取函数
b = C.input                      # 2. 获取输入
b.grad = C.backward(y.grad)      # 3. 调用 backward

B = b.creator
a = B.input
a.grad = B.backward(b.grad)

A = a.creator
x = A.input
x.grad = A.backward(a.grad)

print(x.grad)   # 3.297442541400256

## 7.3 增加 backward 方法（递归实现自动化）

上面"三步循环"每次都一样，可封装进 `Variable.backward`，用**递归**自动完成：

- 取 `creator`；
- 若 `creator` 为 `None`（到达用户输入），停止；
- 否则：给输入变量算 `grad`，再递归调用输入的 `backward`。

于是只需一句 `y.backward()`，反向传播就自动跑完。

In [ ]:
# 7.3 给 Variable 追加 backward 方法（递归）
def backward(self):
    f = self.creator                   # 1. 获取创造自己的函数
    if f is not None:                  # 到达用户输入（creator=None）则停止
        x = f.input                    # 2. 获取函数的输入
        x.grad = f.backward(self.grad) # 3. 计算输入梯度
        x.backward()                   # 递归：对输入再做反向传播

Variable.backward = backward           # 动态追加到 Variable 类

# 重新正向传播，一句话完成自动反向
A = Square(); B = Exp(); C = Square()
x = Variable(np.array(0.5))
a = A(x); b = B(a); y = C(b)

y.grad = np.array(1.0)
y.backward()                           # 自动反向传播！

print(x.grad)   # 3.297442541400256

## 这一步的"为什么"

- **为什么叫 Define-by-Run？** 计算图不是运行前画好的，而是**在正向传播执行的那一刻动态建立**的——函数一执行，`output.set_creator(self)` 就顺手把"连接"接好了。
- **递归为什么能停？** 一路回溯，直到某个变量的 `creator` 是 `None`（用户直接提供的输入），递归自然终止。

## 局限：递归效率低

递归每深入一层都要在调用栈里堆积，嵌套很深时效率低；当前的线性实现也还处理不了"分支 / 复用同一变量"的复杂计算图。

---

> 预告：步骤8 把递归 `backward` 改成**循环**（用 `funcs` 列表 + `pop()`），更高效、也更好扩展。